**DAY 5 - Delta Lake Advanced**

**1.Implement Incremental MERGE**

In [0]:
events_oct_df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv", header = True, inferSchema=True)
events_oct_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.events_oct")

In [0]:
events_nov_df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv", header = True, inferSchema=True)
events_nov_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.events_nov")

In [0]:
#Upsert into a DeltaLake table using Merge

from delta.tables import *

deltaTableEvents = DeltaTable.forName(spark, "workspace.ecommerce.events_oct")
deltaTableEventsUpdates = DeltaTable.forName(spark, "workspace.ecommerce.events_nov")

dfUpdates = deltaTableEventsUpdates.toDF()

deltaTableEvents.alias("events")\
    .merge(
    dfUpdates.alias("updates"),
    "events.user_session = updates.user_session AND events.event_time = updates.event_time"
    ).whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()


In [0]:
%sql
describe workspace.ecommerce.events_oct

In [0]:
#implement Incremental Merge
spark.sql("""
CREATE TABLE IF NOT EXISTS raw_events (
    event_id STRING,
    event_type STRING,
    event_time TIMESTAMP,
    user_id STRING,
    location STRING
)
USING DELTA;
""")



In [0]:
spark.sql("""
INSERT INTO raw_events VALUES
('E001', 'SOS_TRIGGERED', '2026-01-10 21:45:00', 'U1001', 'Delhi_CP'),
('E002', 'SAFE_CHECKIN', '2026-01-10 22:10:00', 'U1002', 'Bangalore_Indiranagar'),
('E003', 'ROUTE_STARTED', '2026-01-10 23:05:00', 'U1003', 'Mumbai_Andheri'),
('E004', 'ROUTE_ENDED', '2026-01-10 23:45:00', 'U1003', 'Mumbai_Bandra'),
('E005', 'SOS_TRIGGERED', '2026-01-11 00:15:00', 'U1004', 'Hyderabad_HitechCity');
""")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS bronze_events (
            event_id STRING,
            event_type STRING,
            event_time TIMESTAMP,
            user_id STRING,
            location STRING,
            created_at TIMESTAMP,
            created_by STRING,
            updated_at TIMESTAMP,
            updated_by STRING,  
            status STRING
        )
USING DELTA;
          """)

In [0]:
%sql
INSERT INTO workspace.default.bronze_events
SELECT
    event_id,
    event_type,
    event_time,
    user_id,
    location,
    current_timestamp() AS created_at,
    'raw_ingestion_job' AS created_by,
    current_timestamp() AS update_at,
    'raw_ingestion_job' as updated_by,
    'ACTIVE' AS status
FROM workspace.default.raw_events;

In [0]:
spark.sql("""
INSERT INTO raw_events VALUES
('E001', 'RESOLVED', '2026-01-12 22:45:00', 'U1001', 'DELHI_CP'),
('E007', 'SAFE_CHECKIN', '2026-01-12 22:10:00', 'U1003', 'Mumbai_Bandra');
""")

In [0]:
%sql
select * from workspace.default.raw_events;



In [0]:
%sql

select * from workspace.default.bronze_events;

In [0]:
%sql

SELECT * FROM raw_events WHERE event_time > (select max(event_time) from bronze_events)

In [0]:
%sql

MERGE INTO bronze_events AS tgt
USING( SELECT * FROM raw_events WHERE event_time > (select max(event_time) from bronze_events)) AS src
ON tgt.event_id = src.event_id
WHEN MATCHED THEN UPDATE 
SET tgt.event_type = src.event_type,
    tgt.event_time = src.event_time,
    tgt.user_id = src.user_id,
    tgt.location = src.location,
    tgt.updated_at = current_timestamp(), 
    tgt.updated_by = 'raw_ingestion_job', 
    tgt.status = 'ACTIVE'
WHEN NOT MATCHED THEN 
INSERT(event_id, event_type, event_time, user_id, location, created_at, created_by, updated_at, updated_by, status) 
VALUES(src.event_id, src.event_type, src.event_time, src.user_id, src.location, current_timestamp(), 'raw_ingestion_job', current_timestamp(), 'raw_ingestion_job', 'ACTIVE')

**2.Query historical versions**

In [0]:
%sql
DESCRIBE HISTORY bronze_events;

In [0]:
%sql
SELECT *
FROM bronze_events VERSION AS OF 0;

In [0]:
%sql
SELECT *
FROM bronze_events VERSION AS OF 1;

In [0]:
%sql 

SELECT * 
FROM bronze_events VERSION AS OF 7;

In [0]:
%sql 

SELECT *
FROM bronze_events
TIMESTAMP AS OF '2026-01-14 03:06:07';

In [0]:
%sql
RESTORE TABLE bronze_events TO VERSION AS OF 2;

**3. OPTIMIZE & ZORDER**

In [0]:
%sql
OPTIMIZE bronze_events
ZORDER BY (event_time, location);

**4.VACUUM for cleanup**

In [0]:
%sql
VACUUM bronze_events RETAIN 168 HOURS; -- 7 days